## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## **Installing and Importing the Necessary Libraries**

In [ ]:
# Installation for GPU llama-cpp-python, run the command with Metal flag for MacOS
!MAKE_ARGS="-DGGML_METAL=on" pip install llama-cpp-python --no-cache-dir

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## **Downloading and Loading the Model**

In [ ]:
!pip install langchain \
  langchain-text-splitters \
  huggingface_hub \
  pandas \
  tiktoken \
  pymupdf \
  langchain \
  langchain-community \
  chromadb \
  sentence-transformers \
  numpy \
  torch

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

import re
from IPython.display import display, Markdown

/opt/homebrew/Caskroom/miniforge/base/envs/rag310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Download the Mistral 7B Instruct v0.1 GGUF model from Hugging Face
# use the Q4_K_M quantization for a good balance of speed and accuracy
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.1-GGUF"
model_basename = "mistral-7b-instruct-v0.1.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename,
)

# Initialize the Llama model
# n_gpu_layers=-1 moves all layers to GPU (if available) for faster inference
# n_ctx=2048 sets the context window size

llm = Llama(
    model_path = model_path,
    n_threads = 2,  # number of CPU threads to use
    n_batch = 512,  # batch size for prompt processing
    n_gpu_layers = -1,  # Offload all layers to GPU
    n_ctx = 2048,  # context window size
    verbose = False  # Verbose mode
)

print(f"Model loaded from: {model_path}")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64 

Model loaded from: /Users/rizwanaqeel/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.1-GGUF/snapshots/731a9fc8f06f5f5e2db8a0cf9d256197eb6e05d1/mistral-7b-instruct-v0.1.Q4_K_M.gguf


## Questions

In [ ]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
query_2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

## Question Answering using LLM

#### Response

In [ ]:
# function to generate and return response from llm
def response(query, max_tokens=256, temperature=0.1, top_p=0.95, top_k=50):
    """
    Generates a response using the loaded LLM without specific system prompting.
    """
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k,
      echo=False # Do not repeat the prompt in the output
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
display(Markdown(f"Question: " + query_1))

answer_1 = response(query_1)

display(Markdown('Answer:'))
display(Markdown(answer_1))

Question: What is the protocol for managing sepsis in a critical care unit?

Answer:



Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis in a critical care unit typically involves the following steps:

1. Early recognition: Sepsis should be suspected in any patient who is critically ill and has a suspected or confirmed infection.
2. Rapid diagnosis: Blood cultures should be obtained as soon as possible to identify the causative pathogen.
3. Appropriate antibiotic therapy: Antibiotics should be started as soon as the causative pathogen is identified.
4. Source control: The source of infection should be identified and treated as soon as possible.
5. Fluid and electrolyte management: Fluid and electrolyte imbalances should be corrected to maintain adequate tissue perfusion and prevent organ dysfunction.
6. Respiratory support: Patients with sepsis may require mechanical ventilation to maintain adequate oxygenation and ventilation.
7. Inotropic and vasopressor support: Inotropic and vasopressor agents may be necessary to maintain adequate cardiac output and blood pressure.
8. Monitoring: Patients with sepsis

#### Observations (LLM – Query 1)
*   Without RAG or specific prompting, the model relies on general training data.
*   It usually provides a generic 'Sepsis Six' or surviving sepsis campaign summary.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
display(Markdown(f"Question: " + query_2))

answer_2 = response(query_2)

display(Markdown('Answer:'))
display(Markdown(answer_2))

Question: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

Answer:

#### Observations (LLM – Query 2)
- **Strength:** The response typically lists common appendicitis symptoms and suggests urgent clinical evaluation.
- **Gap/Risk:** Pure LLM answers may be **over-confident** about “medicine-only cure” vs surgical need without citing a source; may omit diagnostic workup (exam, labs, imaging) or red flags.



### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
display(Markdown(f"Question: " + query_3))

answer_3 = response(query_3)

display(Markdown(f"Answer:"))
display(Markdown(answer_3))

Question: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

Answer:




Sudden patchy hair loss, also known as alopecia areata, is a common hair loss condition that affects both men and women. It is characterized by the appearance of small, round or oval-shaped bald spots on the scalp. The exact cause of alopecia areata is not known, but it is believed to be an autoimmune disorder that affects the hair follicles.

There are several effective treatments and solutions for addressing sudden patchy hair loss. These include:

1. Topical corticosteroids: These are anti-inflammatory medications that can help reduce inflammation in the scalp and promote hair growth. They are available in the form of creams, gels, and shampoos.

2. Minoxidil: This is a medication that is applied to the scalp to stimulate hair growth. It works by increasing blood flow to the hair follicles.

3. Finasteride: This is a medication that is taken orally to slow down hair loss. It works by blocking the production of dihydrotestosterone, a hormone that is thought to contribute to hair loss.

4.

#### Observations (LLM – Query 3)
- **Strength:** Usually identifies likely causes of sudden patchy hair loss and suggests specialist evaluation.
- **Gap/Risk:** May provide broad treatment options without distinguishing evidence level or key differential diagnoses.



### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
display(Markdown(f"Question: " + query_4))

answer_4 = response(query_4)

display(Markdown(f"Answer:"))
display(Markdown(answer_4))

Question: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

Answer:




1. Rest and Recovery: The first step in treating a brain injury is to allow the brain to heal naturally. This may involve rest, sleep, and avoiding activities that could further damage the brain.

2. Medical Treatment: Depending on the severity of the injury, medical treatment may be necessary. This may include surgery, medication, or other treatments to manage symptoms such as swelling, bleeding, or infection.

3. Rehabilitation: Rehabilitation is an important part of the recovery process for brain injuries. This may involve physical therapy, occupational therapy, speech therapy, and other treatments to help restore function and improve quality of life.

4. Supportive Care: Supportive care may be necessary to help manage symptoms and improve overall health. This may include nutritional support, pain management, and other treatments to help manage symptoms such as headaches, dizziness, or fatigue.

5. Psychological Support: Brain injuries can also affect emotional and psychological well-being. Psychological support may be necessary to help manage stress, anxiety, and other emotional symptoms.

6. Cognitive Rehabilitation: Cognitive rehabilitation may be necessary to help improve

#### Observations (LLM – Query 4)
- **Strength:** Typically outlines general TBI management themes (stabilization, imaging, monitoring, rehab).
- **Gap/Risk:** May use generic phrases (“supportive care”) and miss ICU targets/monitoring parameters or neurosurgical escalation criteria.



### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
display(Markdown(f"Question: " + query_5))

answer_5 = response(query_5)

display(Markdown(f"Answer:"))
display(Markdown(answer_5))


Question: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

Answer:





1. Assess the severity of the fracture: The first step is to assess the severity of the fracture. This can be done by examining the leg and feeling for any deformities or misalignments. If the fracture is severe, it may require immediate medical attention.

2. Immobilize the leg: Once the severity of the fracture has been assessed, the leg should be immobilized to prevent further damage. This can be done by using a cast, splint, or crutches.

3. Pain management: Pain management is important to ensure that the person is comfortable during the healing process. Over-the-counter pain relievers such as ibuprofen or acetaminophen can be helpful. If the pain is severe, prescription pain medication may be necessary.

4. Physical therapy: Physical therapy can help to restore strength and flexibility to the leg after a fracture. It is important to follow the physical therapist's instructions carefully to avoid further injury.

5. Recovery time: The recovery time for a fractured leg can vary depending on the severity of the fracture and the individual's overall health. It is important

#### Observations (LLM – Query 5)
- **Strength:** Generally includes first aid basics (immobilization, pain control, transport).
- **Gap/Risk:** May omit compartment syndrome red flags, neurovascular checks, and recovery/rehab considerations; can drift into non-sourced advice.


## Question Answering using LLM with Prompt Engineering

### Prompt Engineering

In [ ]:
# defining prompt + parameter configurations
prompt_configs = [
    {
        "name": "1-Low_temperature_structured",
        "system":(
            "You are an evidence-based medical assistant. "
            "Cite standard clinical guidelines and mention when recommendations "
            "may vary by institution."
        ),
        "style": "bullet_points",
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 50,
    },
    {
        "name": "2-More_elaboration",
        "system":(
            "You are an experienced critical care specialist explaining concepts "
            "to junior residents."
        ),
        "style": "short_paragraphs",
        "temperature": 0.2,
        "top_p": 0.95,
        "top_k": 50,
    },
    {
        "name": "3-Concise_checklist",
        "system": (
            "You create concise checklists that summarize key steps and red flags."
        ),
        "style": "checklist",
        "temperature": 0.1,
        "top_p": 0.9,
        "top_k": 30,
    },
    {
        "name": "4-Patient_safety_focus",
        "system": (
            "You prioritize patient safety, explicit monitoring parameters, and "
            "situations where urgent escalation is needed."
        ),
        "style": "safety_focused",
        "temperature": 0.3,
        "top_p": 0.92,
        "top_k": 60,
    },
    {
        "name": "5-Uncertainty_explicit",
        "system": (
            "You explicitly mention uncertainties, variations in guidelines, and "
            "stress that real clinicians must make final decisions."
        ),
        "style": "with_uncertainty",
        "temperature": 0.2,
        "top_p": 0.95,
        "top_k": 50,
    },
]

### Build Engineered Prompt

In [ ]:
def build_engineered_prompt(question: str, system: str, style: str) -> str:
  style_instruction = {
      "bullet_points": "Answer in 4–8 bullet points.",
      "short_paragraphs": "Answer in 3-5 short paragraphs.",
      "checklist": "Return a short checklist of clearly numbered steps.",
      "safety_focused":(
          "Highlight monitoring, red flags, and when to escalate to higher level of care."
      ),
      "with_uncertainty": (
          "Include a short section on limitations and uncertainty."
      ),
      "scenario_adaptive": (
            "List the Core Medical Principle from the text (e.g., Immobilization).\n"
            "Explain how to apply it in the user's scenario (e.g., Hiking).\n"
            "List specific Red Flags that require immediate evacuation."
        ),
  }.get(style, "Provide a clear, structured answer.")

  # Note: Mistral generally uses [INST]...[/INST], but <<SYS>> tags are often understood
  # as legacy Llama-2 formatting.
  return(
      f"[INST] <<SYS>>{system}<<SYS>>\n\n"
      f"Question: {question}\n"
      f"{style_instruction}\n"
      f"Do not provide personal medical advice; speak in general terms only.\n"
      f"[/INST]"
  )

### Run Engineered Prompt

In [ ]:
def run_prompt_engineering_tests(question_text):
  """
  Iterates through all prompt_configs for a single question to demonstrate
  variety in response style and parameters.
  """

  print(f"### TESTING QUESTION: {question_text}\n" + "="*60)

  for config in prompt_configs:
    print(f"\n--- Configuration: {config['name']} ---")

    # Build prompt
    final_prompt = build_engineered_prompt(
        question=question_text,
        system=config["system"],
        style=config["style"],
    )

    # Call LLM with specific parameters from the config
    output = llm(
        prompt=final_prompt,
        max_tokens=256,
        temperature=config["temperature"],
        top_p=config["top_p"],
        top_k=config["top_k"],
        echo=False
    )

    print(output["choices"][0]["text"].strip())
    print("_" * 30)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
run_prompt_engineering_tests(query_1)

### TESTING QUESTION: What is the protocol for managing sepsis in a critical care unit?

--- Configuration: 1-Low_temperature_structured ---
1. The Surviving Sepsis Campaign (SSC) provides clinical guidelines for managing sepsis in the critical care unit.
2. The SSC recommends early recognition and rapid diagnosis of sepsis, followed by appropriate antimicrobial therapy and fluid resuscitation.
3. The SSC also recommends the use of vasopressors to maintain blood pressure and organ perfusion in severe cases.
4. The SSC advises against the use of corticosteroids in sepsis, except in rare cases of refractory shock.
5. The SSC recommends the use of Sequential Organ Failure Assessment (SOFA) score to assess the severity of illness and guide treatment decisions.
6. The SSC advises against the use of broad-spectrum antibiotics for empirical treatment of sepsis, as this can lead to antibiotic resistance.
7. The SSC recommends the use of targeted antimicrobial therapy based on culture results a

#### Observations (Prompt Engineering – Query 1)
- Checklist/stepwise prompts generally improve **clarity** (actionable steps) compared to baseline.
- Lower temperature (≈0.0–0.1) tends to reduce “fluff” and keep responses closer to clinical framing.
- Remaining limitation: without retrieval, the model may still state details not verifiable from a trusted source.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
run_prompt_engineering_tests(query_2)

### TESTING QUESTION: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

--- Configuration: 1-Low_temperature_structured ---
- Common symptoms of appendicitis include abdominal pain, nausea, vomiting, diarrhea, fever, and loss of appetite.
- The standard treatment for appendicitis is surgical removal of the appendix.
- However, in some cases, antibiotics may be used to treat the infection that caused the appendicitis.
- The decision to use antibiotics instead of surgery may vary by institution and depends on factors such as the severity of the infection and the patient's overall health.
- In some cases, surgery may be delayed if the infection is not severe or if the patient is not a good candidate for surgery.
- The surgical procedure for appendicitis is typically done through a small incision in the abdomen and involves removing the appendix and any damaged tissue.
- The recovery time for app

#### Observations (Prompt Engineering – Query 2)
- Prompts that force separation of **symptoms → diagnosis/workup → treatment → escalation** improve completeness.
- Safety-focused prompts highlight urgency and escalation, but can remain non-specific without source grounding.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
run_prompt_engineering_tests(query_3)

### TESTING QUESTION: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

--- Configuration: 1-Low_temperature_structured ---
1. The American Academy of Dermatology (AAD) recommends that individuals experiencing sudden patchy hair loss should first see a healthcare provider to rule out any underlying medical conditions.
2. The AAD also suggests that individuals may consider using over-the-counter (OTC) topical treatments such as minoxidil or finasteride, which have been shown to be effective in treating hair loss.
3. In some cases, a healthcare provider may prescribe a medication such as corticosteroids or immunosuppressants to address the underlying cause of the hair loss.
4. The AAD recommends that individuals should avoid using harsh chemicals or heat styling tools on their hair, as these can damage the hair follicles and contribute to hair loss.
5

#### Observations (Prompt Engineering – Query 3)
- Structured prompts help separate **possible causes** from **treatments** and avoid mixing them.
- Uncertainty-aware prompts reduce overconfidence in etiology without clinical workup.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
run_prompt_engineering_tests(query_4)

### TESTING QUESTION: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

--- Configuration: 1-Low_temperature_structured ---
1. The American Academy of Neurology (AAN) recommends that individuals with a brain injury receive a comprehensive evaluation by a healthcare provider, including a neurologist, to determine the extent of the injury and develop a treatment plan.
2. The Centers for Disease Control and Prevention (CDC) recommends that individuals with a brain injury receive prompt medical attention, including emergency care if necessary, and follow up with a healthcare provider for ongoing care.
3. The World Health Organization (WHO) recommends that individuals with a brain injury receive appropriate medical care, including surgery if necessary, and follow up with a healthcare provider for ongoing care and rehabilitation.
4. The American College of Surgeons recommends th

#### Observations (Prompt Engineering – Query 4)
- Safety prompts improve coverage of stabilization/monitoring and highlight escalation triggers.
- Without RAG, protocol-level parameters may still be generic or inconsistent.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
run_prompt_engineering_tests(query_5)

### TESTING QUESTION: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

--- Configuration: 1-Low_temperature_structured ---
1. Immobilization: The first step in treating a leg fracture is to immobilize the affected area using a cast, splint, or brace to prevent further damage and promote healing.
2. Pain management: Pain relief should be provided through a combination of medication, physical therapy, and other interventions.
3. Surgery: In some cases, surgery may be necessary to realign the broken bone and secure it in place with screws, plates, or pins.
4. Rehabilitation: Physical therapy and rehabilitation exercises are essential to restore strength, flexibility, and range of motion to the affected leg.
5. Follow-up care: Regular follow-up appointments with a healthcare provider are necessary to monitor progress, adjust treatment plans, and address any compl

#### Observations (Prompt Engineering – Query 5)
- Checklist prompts improve readability for emergency scenarios (immobilization, transport, monitoring).
- Remaining limitation: details vary by guideline; retrieval is needed to ensure source-consistent steps.


## Data Preparation for RAG

### Loading the Data

In [ ]:
# load the PDF file
loader = PyMuPDFLoader("/Users/rizwanaqeel/Documents/AI-ML/NLP-Assignment/medical_diagnosis_manual.pdf")
docs = loader.load()

print(f"Total pages loaded: {len(docs)}")

Total pages loaded: 4114


#### Mask email addresses in the loaded documents

In [ ]:
#  Full emails (normal case)
full_email = re.compile(r"\b[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9.-]+\b")

#  Fragments like "name@h", "name@hshing", "name@whatever" (no dot)
email_fragment = re.compile(r"\b[a-zA-Z0-9_.+-]{2,}@[a-zA-Z0-9-]{1,}\b")

def redact_emails(text, mask="[REDACTED_EMAIL]"):
    # normalize line breaks that split tokens
    normalized = text.replace("\n", "")

    # redact full emails
    normalized = full_email.sub(mask, normalized)

    # redact fragments that still contain "@"
    normalized = email_fragment.sub(mask, normalized)

    return normalized

# apply to all documents
for doc in docs:
    doc.page_content = redact_emails(doc.page_content)

### Data Overview

#### Checking the first 5 pages

In [ ]:
# Display content of the first 5 pages to verify loading
for i in range(5):
  print(f"--- Page {i+1} ---")
  print(docs[i].page_content[:500])   # Print first 500 chars per page
  print("\n")

--- Page 1 ---
[REDACTED_EMAIL] for personal use by [REDACTED_EMAIL] the contents in part or full is liable


--- Page 2 ---
[REDACTED_EMAIL] file is meant for personal use by [REDACTED_EMAIL] only.Sharing or publishing the contents in part or full is liable for legal action.


--- Page 3 ---
Table of Contents1Front    ................................................................................................................................................................................................................1Cover    .......................................................................................................................................................................................................2Front Matter    .......................................


--- Page 4 ---
491Chapter 44. Foot & Ankle Disorders    .....................................................................................................................................502Chapter 45.

#### Checking the number of pages

In [ ]:
print(f"Number of pages available: {len(docs)}")

Number of pages available: 4114


### Data Chunking

In [ ]:
# Split the text into smaller chunks for embedding
# chunk_size=1000: Good balance for capturing context without exceeding token limits
# chunk_overlap=200: Ensures context isn't lost at the boundaries of chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

print(f"Total text chunks created: {len(chunks)}")
print(f"Sample chunk: {chunks[0].page_content[:500]}")

Total text chunks created: 17641
Sample chunk: [REDACTED_EMAIL] for personal use by [REDACTED_EMAIL] the contents in part or full is liable


### Embedding

In [ ]:
# Initialize Sentence Transformer Embeddings
# 'all-MiniLM-L6-v2' is a robust, lightweight model for semantic search
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/var/folders/0z/c1z6cjy92ds1jj6cjb84glxm0000gn/T/ipykernel_78604/1293159433.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


### Vector Database

In [ ]:
# Create the Vector Store using ChromaDB
# This converts text chunks into vectors and stores them for retrieval
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="medical_manual_collection",
    persist_directory="./chroma_db"   # Persist data to disk
)

print(f"Vector Database created successfully.")

Vector Database created successfully.


### Retriever

In [ ]:
# Configure the retriever
# search_kwargs={"k": 3} means we retrieve the top 3 most relevant chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

### System and User Prompt Template

In [ ]:
# Define the RAG prompts
qna_system_message = (
    "You are a helpful and accurate medical assistant."
    "Use the provided Merck Manual context to answer the user's question."
    "If the answer is not present in the context, state that you do not know."
    "Do not fabricate information."
)

qna_user_message_template = """
CONTEXT (Merck Manual excerpts):
{context}

QUESTION:
{question}

INSTRUCTIONS:
- Answer using ONLY the CONTEXT.
- Return Markdown with this exact structure:

## Summary (2–3 lines)
## Key steps
- ...
## Monitoring & targets
- ...
## Red flags / escalate
- ...
## Notes / limitations (if context is missing)
- ...

Now write the answer.
"""

### Response Function

In [ ]:
def generate_rag_response(
    user_input,
    k=4,
    max_tokens=256,
    temperature=0.0,
    top_p=0.95,
    top_k=50,
):
    """
    Retrieve relevant chunks from the Merck Manual and generate an answer
    conditioned on those chunks.
    """
    global qna_system_message, qna_user_message_template, vector_db

    # Build a retriever with the desired k
    retriever = vector_db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

    # get relevant documents
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine chunks into a single context string
    context_for_query = "\n\n".join(context_list)

    user_message = qna_user_message_template.format(
        context=context_for_query,
        question=user_input,
    )

    prompt = qna_system_message + "\n\n" + user_message

    try:
        raw = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )
        answer = raw["choices"][0]["text"].strip()
    except Exception as e:
        answer = f"Sorry, an error occurred while generating the answer:\n{e}"

    return answer, context_for_query


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
print(f"Question: {query_1}\n")
rag_ans_1, rag_ctx_1 = generate_rag_response(query_1)
display(Markdown(rag_ans_1))

Question: What is the protocol for managing sepsis in a critical care unit?



## Summary
Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit.

## Key steps
1. Recognize the signs and symptoms of sepsis, including fever, tachycardia, hypotension, and altered mental status.
2. Obtain a blood culture and initiate empirical antibiotic therapy.
3. Monitor the patient's vital signs, laboratory values, and fluid balance.
4. Administer vasopressors and fluid resuscitation as needed to maintain adequate blood pressure and tissue perfusion.
5. Consider the use of anticoagulation therapy to prevent disseminated intravascular coagulation.
6. Monitor the patient for signs of organ dysfunction or failure, and adjust treatment accordingly.

## Monitoring & targets
- Monitor the patient's vital signs (blood pressure, heart rate, respiratory rate, temperature) every 15 minutes.
- Obtain daily blood cultures and monitor for signs of infection.
- Monitor the patient's serum lactate level and adjust fluid resuscitation and vasopressor therapy as needed.
- Monitor the patient'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
print(f"Question: {query_2}\n")
rag_ans_2, rag_ctx_2 = generate_rag_response(query_2)
display(Markdown(rag_ans_2))

Question: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?



## Summary
Appendicitis is a medical condition that can cause severe pain in the right lower quadrant of the abdomen. The classic symptoms include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. If left untreated, it can lead to necrosis, gangrene, and perforation. There is no cure for appendicitis via medicine, and the only treatment is surgical removal of the appendix.

## Key steps
1. Recognize the symptoms of appendicitis, which include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia.
2. If the symptoms persist or worsen, seek medical attention immediately.
3. The diagnosis of appendicitis is typically made through a physical examination and imaging tests such as ultrasound or CT scan.
4. Treatment for appendicitis involves surgical removal of the appendix.

## Monitoring & targets
1. Monitor for any signs of worsening symptoms, such as severe pain, fever, or vomiting.
2. Ensure that the patient is receiving

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
print(f"Question: {query_3}\n")
rag_ans_3, rag_ctx_3 = generate_rag_response(query_3)
display(Markdown(rag_ans_3))

Question: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?



## Summary
Sudden patchy hair loss, commonly seen as localized bald spots on the scalp, can be caused by various factors such as alopecia areata, traction alopecia, tinea capitis, trichotillomania, and scarring alopecia. Treatment options vary depending on the underlying cause.

## Key steps
1. Identify the cause of the hair loss by examining the scalp and consulting a healthcare professional.
2. Treat the underlying cause with appropriate medications or therapies.
3. Consider surgical options such as follicle transplant, scalp flaps, or alopecia reduction if the patient is self-conscious about their hair loss.

## Monitoring & targets
Monitor the patient's response to treatment and adjust the treatment plan as needed.

## Red flags / escalate
If the hair loss persists or worsens despite treatment, consult a healthcare professional for further evaluation and management.

## Notes / limitations
The context does not provide specific information on the effectiveness of certain treatments for sudden patchy hair loss. It is important to note that each patient's response to treatment may vary.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
print(f"Question: {query_4}\n")
rag_ans_4, rag_ctx_4 = generate_rag_response(query_4)
display(Markdown(rag_ans_4))

Question: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?



## Summary
Rehabilitation is recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.

## Key steps
Rehabilitation is best provided through a team approach that combines physical, occupational, and speech therapy, skill-building activities, and counseling to meet the patient's social and emotional needs. Brain injury support groups may provide assistance to the families of brain-injured patients. For patients whose coma exceeds 24 h, a prolonged period of rehabilitation, particularly in cognitive and emotional areas, is often required. Rehabilitation services should be planned early.

## Monitoring & targets
Rehabilitation services should be monitored regularly to ensure that the patient is making progress and achieving their goals. The target for rehabilitation is to improve the patient's functional abilities and quality of life.

## Red flags / escalate
If the patient's condition worsens or if they experience any complications during rehabilitation, the rehabilitation team should be notified immediately. The rehabilitation team may need to adjust the treatment plan or refer the patient to a specialist for further evaluation and

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
print(f"Question: {query_5}\n")
rag_ans_5, rag_ctx_5 = generate_rag_response(query_5)
display(Markdown(rag_ans_5))

Question: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?



## Summary

The provided context does not contain information about fractured legs during hiking trips.

## Key steps

The provided context does not contain information about fractured legs during hiking trips.

## Monitoring & targets

The provided context does not contain information about fractured legs during hiking trips.

## Red flags / escalate

The provided context does not contain information about fractured legs during hiking trips.

## Notes / limitations

The provided context does not contain information about fractured legs during hiking trips.

### Fine-tuning

#### Define Configurations and fine tunning Logic

In [ ]:
# ---------------------------
# RAG Fine Tuning: Definitions
# ---------------------------

# Define the list of questions to test for all configurations
questions = [
    "What is the protocol for managing sepsis in a critical care unit?",
    "What are the common symptoms of appendicitis, and can it be cured via medicine?",
    "What are the effective treatments for sudden patchy hair loss?",
    "What treatments are recommended for physical injury to brain tissue?",
    "What are the necessary precautions for a fractured leg during a hiking trip?"
]

# Define the configurations
rag_configs = [
    {"name": "config1_chunks_512_overlap_64_k4_temp0", "chunk_size": 512, "chunk_overlap": 64, "k": 4, "temperature": 0.0},
    {"name": "config2_chunks_1500_overlap_200_k3_temp0", "chunk_size": 1500, "chunk_overlap": 200, "k": 3, "temperature": 0.0},
    {"name": "config3_chunks_1024_overlap_128_k6_temp0", "chunk_size": 1024, "chunk_overlap": 128, "k": 6, "temperature": 0.0},
    {"name": "config4_chunks_1024_overlap_128_k4_temp02", "chunk_size": 1024, "chunk_overlap": 128, "k": 4, "temperature": 0.2},
    {"name": "config5_chunks_800_overlap_100_k5_temp01", "chunk_size": 800, "chunk_overlap": 100, "k": 5, "temperature": 0.1},
]

In [ ]:
def run_rag_experiment(config, questions, source_docs):
    """
    Runs a SINGLE RAG experiment: rebuilds vector store with specific chunking,
    and answers the provided questions.
    """
    global qna_system_message, qna_user_message_template

    print(f"\n{'='*40}\nRunning Config: {config['name']}\n{'='*40}")
    print(f"Params: Chunk={config['chunk_size']}, Overlap={config['chunk_overlap']}, k={config['k']}, Temp={config['temperature']}")

    # DATA PREPARATION (Re-chunking)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"]
    )
    chunks = text_splitter.split_documents(source_docs)
    print(f"-> Created {len(chunks)} chunks.")

    # VECTOR STORE CREATION
    # We use a unique collection name to keep experiments separate
    collection_name = f"rag_{config['name']}"


    # Initialize Chroma with specific collection
    vector_db_exp = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name
    )

    # RETRIEVER DEFINITION
    retriever_exp = vector_db_exp.as_retriever(search_kwargs={"k": config["k"]})

    # ANSWER GENERATION

    for i, q in enumerate(questions):
        print(f"\n--- Question {i+1} ---")
        print(f"Q: {q}")

        # Retrieve
        relevant_docs = retriever_exp.invoke(q)
        context_text = ". ".join([d.page_content for d in relevant_docs])

        # Construct Prompt
        user_message = qna_user_message_template.replace('{context}', context_text)
        user_message = user_message.replace('{question}', q)

        full_prompt = qna_system_message + "\n\n" + user_message

        # Generate
        response = llm(
            prompt=full_prompt,
            max_tokens=256,
            temperature=config["temperature"],
            top_p=0.95,
            top_k=50,
            echo=False
        )

        display(Markdown(response['choices'][0]['text'].strip()))

    # Clear memory
    vector_db_exp = None

#### Execute the Fine-Tuning


##### Experiment 1 (Small Chunks)



In [ ]:
# Experiment 1: Small chunks (512), tight overlap, k=4
# Hypothesis: Good for specific fact lookup, might miss broader context
current_config = rag_configs[0]
run_rag_experiment(current_config, questions, docs)


Running Config: config1_chunks_512_overlap_64_k4_temp0
Params: Chunk=512, Overlap=64, k=4, Temp=0.0
-> Created 31206 chunks.

--- Question 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?


## Summary

Sepsis is a critical condition that requires immediate treatment in an ICU. The protocol for managing sepsis includes aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues, drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C.

## Key steps

1. Monitor the patient frequently for systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, and electrolyte levels, renal function, and sublingual PCO2.
2. Measure urine output, usually with an indwelling catheter.
3. Administer IV antibiotics (eg, ampicillin plus gentamicin) after appropriate cultures are taken and continue until bacterial sensitivity is known.
4. If clinical response is adequate, continue IV therapy until the patient is afebrile for 24 to 48 hours, followed by oral therapy for 4 weeks.
5. Consider adjunctive therapies such as NSAIDs, α-blockers


--- Question 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine?


## Summary
Appendicitis is a medical condition characterized by epigastric or periumbilical pain followed by abdominal pain. It can be cured via surgical removal of the appendix.

## Key steps
1. Diagnosis is clinical, often supplemented by CT or ultrasound.
2. Treatment is surgical removal.

## Monitoring & targets
There are no specific monitoring or targets for appendicitis.

## Red flags / escalate
If the perforation is contained by the omentum, an appendiceal abscess results.

## Notes / limitations
The provided context does not mention any medication that can cure appendicitis.


--- Question 3 ---
Q: What are the effective treatments for sudden patchy hair loss?


## Summary
Hair loss due to other causes is treated. Multiple treatment options for alopecia areata exist and include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).

## Key steps
1. Identify the underlying cause of the hair loss.
2. Treat the underlying cause if possible.
3. If the underlying cause cannot be treated, consider treatment options for alopecia areata.

## Monitoring & targets
Monitor the patient's response to treatment and adjust the treatment plan as needed.

## Red flags / escalate
If the patient's hair loss is severe or does not respond to treatment, consider referral to a dermatologist.

## Notes / limitations
The provided context does not specifically address sudden patchy hair loss. However, the context does provide information on effective treatments for alopecia areata, which may be applicable to sudden patchy hair loss caused by alopecia


--- Question 4 ---
Q: What treatments are recommended for physical injury to brain tissue?


## Summary

Treatment for physical injury to brain tissue includes surgical excision, radiation therapy, and chemotherapy for some types of brain tumors. Injuries are treated with rest, analgesics, and muscle-relaxing drugs with or without surgery until swelling and local pain have subsided.

## Key steps

1. Surgical excision: This is recommended for anaplastic astrocytomas and glioblastomas to reduce tumor mass.
2. Radiation therapy: This is used after surgery to target the tumor and spare normal brain tissue.
3. Chemotherapy: This is used for some types of brain tumors.

## Monitoring & targets

1. Monitor for tumor size and progression.
2. Monitor for neurologic function.

## Red flags / escalate

1. If the injury is severe, immediate medical attention is necessary.
2. If the injury is unstable, immobilization is required to ensure proper alignment.

## Notes / limitations

1. The use of antibiotic prophylaxis for head injuries is controversial due to limited data on its efficacy and


--- Question 5 ---
Q: What are the necessary precautions for a fractured leg during a hiking trip?


## Summary

During a hiking trip, a fractured leg should be treated with splinting and rest, ice, compression, and elevation (RICE). If the fracture is severe, definitive treatment such as reduction may be necessary. It is important to evaluate the patient for potential complications such as hemorrhagic shock, nerve injuries, and fat embolism. If the mechanism suggests potentially severe or multiple injuries, the patient should be evaluated from head to toe for serious injuries to all organ systems and resuscitated.

## Key steps

1. Apply splinting to the fractured leg.
2. Rest, ice, compress, and elevate the leg.
3. If the fracture is severe, consider definitive treatment such as reduction.

## Monitoring & targets

Monitor the patient for signs of complications such as hemorrhagic shock, nerve injuries, and fat embolism.

## Red flags / escalate

If the patient develops signs of hemorrhagic shock, such as rapid heart rate, low blood pressure, or decreased urine output, seek immediate medical attention. If the patient develops signs of nerve injury

##### Experiment 2 (Large Chunks)

In [ ]:
# Experiment 2: Large chunks (1500), k=3
# Hypothesis: Better for complex protocols like Sepsis that span many paragraphs
current_config = rag_configs[1]
run_rag_experiment(current_config, questions, docs)


Running Config: config2_chunks_1500_overlap_200_k3_temp0
Params: Chunk=1500, Overlap=200, k=3, Temp=0.0
-> Created 11789 chunks.

--- Question 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?


## Summary

Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis includes fluid resuscitation with 0.9% normal saline, administration of oxygen, broad-spectrum antibiotics, drainage of abscesses and excision of necrotic tissue, normalization of blood glucose levels, and replacement-dose corticosteroids. Patients with septic shock should be treated in an ICU and monitored frequently for systemic pressure, CVP, PAOP, pulse oximetry, ABGs, blood glucose, lactate, and electrolyte levels, renal function, urine output, and sublingual PCO2. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria with hypotension is not a contraindication to vigorous fluid resuscitation. The quantity of fluid required often far exceeds the normal blood volume and may reach 1


--- Question 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine?


## Summary
Appendicitis is a condition that can cause epigastric or periumbilical pain, nausea, vomiting, anorexia, and low-grade fever. It can be cured via surgery, which involves removing the appendix.

## Key steps
1. If you suspect appendicitis, seek medical attention immediately.
2. The surgeon will likely perform an open or laparoscopic appendectomy.
3. Antibiotics may be prescribed before and after surgery to prevent infection.

## Monitoring & targets
1. Monitor for signs of perforation, such as severe abdominal pain, nausea, vomiting, and fever.
2. If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or continued for a fixed course, according to the surgeon's preference.

## Red flags / escalate
1. If the patient's symptoms worsen or if there are signs of perforation, seek immediate medical attention.

## Notes / limitations
- The classic symptoms of appendicitis are not present in < 5


--- Question 3 ---
Q: What are the effective treatments for sudden patchy hair loss?


## Summary

Sudden patchy hair loss can be treated with warm compresses and manual removal of ingrown hairs with a needle or tweezers. Topical hydrocortisone 1% or topical antibiotics can be used for mild inflammation. Oral tetracycline (250 to 500 mg qid) or oral erythromycin (250 to 500 mg qid, 333 mg tid, 500 mg bid) can be used for moderate to severe inflammation. Tretinoin (retinoic acid) liquid or cream or benzoyl peroxide cream may also be effective in mild or moderate cases but may irritate the skin. Topical eflornithine hydrochloride cream may help by slowing hair growth. Hairs should be allowed to grow out; grown hair can then be cut to about 0.5 cm length. Depilatories are an alternative but may irritate the skin. Hair follicles can be permanently removed by electrolysis or laser treatment.

## Key steps

1. Warm compresses and manual removal of ing


--- Question 4 ---
Q: What treatments are recommended for physical injury to brain tissue?


## Summary

Treatment for physical injury to brain tissue includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. Rehabilitation is often required after the initial treatment.

## Key steps

1. Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure.
2. Surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
3. Rehabilitation is often required after the initial treatment.

## Monitoring & targets

Monitoring should include maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium.

## Red flags / escalate

If brain tumors are expected to soon be fatal, end-of-life issues should be considered.

## Notes / limitations

The provided context does not mention any specific treatments for physical injury to brain tissue


--- Question 5 ---
Q: What are the necessary precautions for a fractured leg during a hiking trip?


## Summary

During a hiking trip, if a leg is fractured, the necessary precautions include immediate medical attention, immobilization of the affected limb, and pain control.

## Key steps

1. Immediate medical attention: If a leg is fractured during a hiking trip, seek immediate medical attention.
2. Immobilization: The affected limb should be immobilized to prevent further injury and promote healing. This can be done with a cast, splint, or sling, depending on the severity of the fracture.
3. Pain control: Pain control should be provided with analgesics, typically acetaminophen or NSAIDs. If pain persists for more than 72 hours after a seemingly minor injury, referral to a specialist is recommended.

## Monitoring & targets

Monitor the affected limb for signs of infection, such as redness, swelling, or drainage. If any of these signs are present, seek immediate medical attention.

## Red flags / escalate

If the fracture is severe and requires surgery, it is important to follow the surgeon's instructions for postoperative care, including

####

##### Experiment 3 (High Retrieval)

In [ ]:
# Experiment 3: Standard chunks (1024), High retrieval (k=6)
# Hypothesis: More context retrieved, but risk of "lost in the middle" or noise
current_config = rag_configs[2]
run_rag_experiment(current_config, questions, docs)


Running Config: config3_chunks_1024_overlap_128_k6_temp0
Params: Chunk=1024, Overlap=128, k=6, Temp=0.0
-> Created 16290 chunks.

--- Question 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?


## Summary

Sepsis is a critical condition that requires aggressive management in a critical care unit. The mainstay of treatment is supportive care, including fluid resuscitation, antibiotics, and surgical intervention. The protocol for managing sepsis involves close monitoring of vital signs, laboratory values, and organ function. Red flags should be closely watched for, and escalation of care may be necessary.

## Key steps

1. Administer fluid resuscitation to maintain adequate tissue perfusion.
2. Initiate antibiotic therapy to target the causative pathogen.
3. Perform surgical intervention to remove infected or necrotic tissues.
4. Monitor and manage organ dysfunction.
5. Administer corticosteroids and activated protein C as needed.

## Monitoring & targets

1. Monitor vital signs, including heart rate, blood pressure, respiratory rate, and temperature.
2. Monitor laboratory values, including white blood cell count, serum creatinine, and arterial blood gas.
3. Monitor organ function using appropriate tests, such as urine output and serum bilirubin.
4. Target blood pressure to maintain


--- Question 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine?


## Summary
Appendicitis is acute inflammation of the vermiform appendix, typically resulting in abdominal pain, anorexia, and abdominal tenderness. Diagnosis is clinical, often supplemented by CT or ultrasound. Treatment is surgical removal.

## Key steps
1. Diagnosis: Clinical evaluation, often supplemented by CT or ultrasound.
2. Treatment: Surgical removal of the appendix.

## Monitoring & targets
- Monitor urine output with a catheter.
- Fluid status is maintained by adequate IV fluid and electrolyte replacement.

## Red flags / escalate
- If the perforation is contained by the omentum, an appendiceal abscess results.
- If surgery is impossible, antibiotics—although patients may improve when only some of the precipitating factors are corrected.

## Notes / limitations
- The appendix may also be affected by Crohn's disease or ulcerative colitis with pancolitis.
- Other conditions affecting the appendix include carcinoids, cancer, villous adenomas, and divert


--- Question 3 ---
Q: What are the effective treatments for sudden patchy hair loss?


## Summary

Sudden patchy hair loss can be treated with a long-acting oral tetracycline in combination with a potent topical corticosteroid.

## Key steps

1. Diagnose the underlying cause of the hair loss.
2. Treat the hair loss with a long-acting oral tetracycline in combination with a potent topical corticosteroid.

## Monitoring & targets

Monitor the patient's response to treatment and adjust the treatment plan as needed.

## Red flags / escalate

If the patient's hair loss is severe or does not improve with treatment, consider referral to a dermatologist.

## Notes / limitations

If the patient's hair loss is due to an underlying medical condition, such as alopecia areata or androgenetic alopecia, additional treatment may be necessary.


--- Question 4 ---
Q: What treatments are recommended for physical injury to brain tissue?


## Summary

Treatment for traumatic brain injury (TBI) includes prevention of secondary disabilities, prevention of pneumonia, and family education. Early intervention by rehabilitation specialists is indispensable for maximal functional recovery. Surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.

## Key steps

1. Prevention of secondary disabilities
2. Prevention of pneumonia
3. Family education
4. Early intervention by rehabilitation specialists
5. Surgery to place monitors, treat intracranial pressure, decompress the brain, or remove intracranial hematomas
6. Maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium in the first few days after the injury
7. Rehabilitation in the subsequent period

## Monitoring & targets

Monitoring should focus on maintaining


--- Question 5 ---
Q: What are the necessary precautions for a fractured leg during a hiking trip?


## Summary

During a hiking trip, a fractured leg should be treated with RICE (Rest, Ice, Compression, Elevation) and splinting to prevent further injury and reduce swelling. If the injury is severe, it may be necessary to seek medical attention for evaluation and treatment.

## Key steps

1. Rest: Avoid putting weight on the injured leg to prevent further injury.
2. Ice: Apply ice to the injured area for 20 minutes at a time to reduce swelling and inflammation.
3. Compression: Wrap the injured area with an elastic bandage to reduce edema and pain.
4. Elevation: Elevate the injured area above heart level to facilitate drainage of fluid and reduce swelling.
5. Splinting: Apply a splint to the injured leg to keep it immobile and prevent further injury.

## Monitoring & targets

Monitor the injured leg for signs of infection, such as redness, warmth, or drainage. If the injury is severe, seek medical attention for evaluation and treatment.

## Red flags / escalate

If the injury is severe and cannot be managed

##### Experiment 4 (Higher Creativity)

In [ ]:
# Experiment 4: Standard chunks (1024), Higher Temperature (0.2)
# Hypothesis: Responses should be less robotic/repetitive
current_config = rag_configs[3]
run_rag_experiment(current_config, questions, docs)


Running Config: config4_chunks_1024_overlap_128_k4_temp02
Params: Chunk=1024, Overlap=128, k=4, Temp=0.2
-> Created 16290 chunks.

--- Question 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?


## Summary

Sepsis is a critical condition that requires aggressive management in a critical care unit. The mainstay of treatment is supportive care, including fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues, and drainage of pus. The severity of sepsis is classified into three categories: sepsis, severe sepsis, and septic shock. Treatment for severe sepsis and septic shock includes intensive control of blood glucose and administration of corticosteroids and activated protein C.

## Key steps

1. Diagnose sepsis based on clinical signs such as fever, hypotension, oliguria, and confusion.
2. Administer fluid resuscitation to maintain tissue perfusion.
3. Administer antibiotics to treat the underlying infection.
4. Perform surgical excision of infected or necrotic tissues and drainage of pus.
5. Monitor and manage organ failure.
6. Monitor and manage blood glucose levels.
7. Administer corticosteroids and activated protein C if necessary.

## Monitoring & targets


--- Question 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine?


## Summary
Appendicitis is a common condition that can cause abdominal pain, anorexia, and abdominal tenderness. It can be cured via surgical removal of the appendix.

## Key steps
1. Diagnose appendicitis based on clinical symptoms and signs.
2. Monitor urine output with a catheter to maintain fluid status.
3. Administer IV antibiotics effective against intestinal flora.
4. Perform surgery to remove the appendix.

## Monitoring & targets
- Monitor urine output with a catheter to maintain fluid status.
- Administer IV antibiotics effective against intestinal flora.

## Red flags / escalate
- If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized or continued for a fixed course, according to the surgeon's preference.

## Notes / limitations
- The Merck Manual does not provide information on whether appendicitis can be cured via medicine.


--- Question 3 ---
Q: What are the effective treatments for sudden patchy hair loss?


## Summary

Sudden patchy hair loss can be treated with topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).

## Key steps

1. Evaluate the patient's history of present illness, including the onset and duration of hair loss, whether hairshedding is increased, and whether hair loss is generalized or localized.
2. Assess associated symptoms such as pruritus and scaling.
3. Inquire about typical hair care practices, including use of braids, rollers, and hair dryers, and whether they routinely pull or twist their hair.
4. Conduct a review of systems to identify recent exposures to noxious stimuli (eg, drugs, toxins, radiation) and stressors (eg, surgery, chronic illness, fever, psychologic stressors).
5. Symptoms of possible causes should be sought, including fatigue and cold intolerance (hyp


--- Question 4 ---
Q: What treatments are recommended for physical injury to brain tissue?


## Summary
Traumatic brain injury (TBI) is a common cause of death and disability. Diagnosis is suspected clinically and confirmed by imaging. Initial treatment consists of ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.

## Key steps
- Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure.
- Place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
- Maintain adequate brain perfusion and oxygenation and prevent complications of altered sensorium.
- Provide rehabilitation.

## Monitoring & targets
- Monitor intracranial pressure and adjust treatment accordingly.


--- Question 5 ---
Q: What are the necessary precautions for a fractured leg during a hiking trip?


## Summary

During a hiking trip, a fractured leg should be treated with the RICE (Rest, Ice, Compression, Elevation) method to prevent further injury and reduce swelling. The fracture should be evaluated by a healthcare professional and may require reduction (realignment of fracture fragments or dislocated joints) or imaging studies. Infections should be monitored and treated promptly if they occur.

## Key steps

1. Rest: Avoid putting weight on the injured leg to prevent further injury.
2. Ice: Apply ice to the injured area for 20 minutes at a time, wrapped in plastic or a towel to prevent direct contact with the skin.
3. Compression: Wrap the injured area with an elastic bandage to reduce edema and pain.
4. Elevation: Keep the injured area above heart level to facilitate fluid drainage and reduce swelling.
5. Evaluation: Seek medical evaluation for the fracture and any potential complications.

## Monitoring & targets

Monitor the injured leg for signs of infection, such as redness, warmth, or drainage. If an infection occurs, seek prompt medical

##### Experiment 5 (Balanced)

In [ ]:
# Experiment 5: Moderate chunks (800), Moderate k(5), Slight Temp (0.1)
# Hypothesis: A balanced approach attempting to get the best of both worlds
current_config = rag_configs[4]
run_rag_experiment(current_config, questions, docs)


Running Config: config5_chunks_800_overlap_100_k5_temp01
Params: Chunk=800, Overlap=100, k=5, Temp=0.1
-> Created 20594 chunks.

--- Question 1 ---
Q: What is the protocol for managing sepsis in a critical care unit?


## Summary

Sepsis is a critical condition that requires aggressive management in a critical care unit. The protocol for managing sepsis includes fluid resuscitation, antibiotics, drainage of abscesses and excision of necrotic tissue, normalization of blood glucose levels, and replacement-dose corticosteroids. Patients with septic shock should be treated in an ICU and monitored frequently for systemic pressure, CVP, PAOP, or both; pulse oximetry; ABGs; blood glucose, lactate, and electrolyte levels; renal function, and possibly sublingual PCO2. Urine output, a good control of blood glucose and administration of corticosteroids and activated protein C. A spectrum of severity exists, and the risk-benefit assessment is required in other patients with increased risk of serious bleeding.

## Key steps

1. Fluid resuscitation with 0.9% normal saline
2. Broad-spectrum antibiotics (modified by culture results)
3. Drainage of abscesses and excision of necrotic tissue
4


--- Question 2 ---
Q: What are the common symptoms of appendicitis, and can it be cured via medicine?


## Summary
Appendicitis is a common cause of acute abdominal pain requiring surgery. The classic symptoms include epigastric or periumbilical pain followed by nausea, vomiting, and anorexia. If left untreated, it can lead to necrosis, gangrene, and perforation.

## Key steps
1. Diagnose appendicitis through clinical examination and imaging studies such as CT or ultrasound.
2. Treat with surgical removal of the appendix.

## Monitoring & targets
- Monitor for signs of perforation, such as severe abdominal pain, nausea, vomiting, and fever.
- Target for prompt surgical intervention to prevent complications.

## Red flags / escalate
- If the perforation is contained by the omentum, an appendiceal abscess may occur.
- If the patient presents with severe abdominal pain, nausea, vomiting, and fever, consider urgent surgical intervention.

## Notes / limitations
- Appendicitis is a common cause of acute abdominal pain requiring surgery.
- The classic symptoms include


--- Question 3 ---
Q: What are the effective treatments for sudden patchy hair loss?


## Summary

Sudden patchy hair loss can be treated with various options depending on the underlying cause.

## Key steps

1. Determine the cause of the hair loss through a microscopic hair examination or scalp biopsy.
2. Treat the underlying cause if present, such as androgenetic alopecia, alopecia areata, traction alopecia, or scarring alopecia.
3. Consider topical or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy, or PUVA for alopecia areata.
4. Eliminate physical traction or stress to the scalp for traction alopecia.
5. Treat scarring alopecia with a long-acting oral tetracycline in combination with a potent topical corticosteroid.
6. Treat lichen planopilaris and chronic cutaneous lupus lesions with oral antimalarials, corticosteroids, retinoids, or immunosuppressants.

## Monitoring & targets

Monitor the effectiveness of


--- Question 4 ---
Q: What treatments are recommended for physical injury to brain tissue?


## Summary

Treatment for physical injury to brain tissue, also known as traumatic brain injury (TBI), includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. Rehabilitation is also important for many patients, with physical and occupational therapy being useful for making the environment safer and providing devices that help patients circumvent the primary deficit.

## Key steps

1. Ensure a reliable airway and maintain adequate ventilation, oxygenation, and blood pressure.
2. Surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas.
3. Rehabilitation with physical and occupational therapy may be necessary to improve functioning and make the environment safer.

## Monitoring & targets

Monitor intracranial pressure and ensure adequate brain perfusion and oxygenation.

## Red flags / escalate

If


--- Question 5 ---
Q: What are the necessary precautions for a fractured leg during a hiking trip?


## Summary

The provided context does not contain information about necessary precautions for a fractured leg during a hiking trip.

## Key steps

The provided context does not contain information about necessary key steps for a fractured leg during a hiking trip.

## Monitoring & targets

The provided context does not contain information about necessary monitoring and targets for a fractured leg during a hiking trip.

## Red flags / escalate

The provided context does not contain information about necessary red flags or escalation for a fractured leg during a hiking trip.

## Notes / limitations

The provided context does not contain information about necessary notes or limitations for a fractured leg during a hiking trip.



* **Chunk size strongly affects clinical fidelity:** Larger or moderately sized chunks (800–1500 chars) preserved protocol-level details (e.g., ICU monitoring targets) better than very small chunks, which fragmented context.
* **High retrieval count (k) increases error risk:** Raising k (e.g., k=6) led to retrieval contamination, where unrelated treatment snippets were blended into answers (most evident in the hair-loss question).
* **Lower temperature improves medical grounding:** Temperatures above 0.1 introduced paraphrasing and “helpful-sounding” additions that were not strictly supported by retrieved text.
* **Protocol-heavy questions benefit from fewer, coherent chunks:** Sepsis and TBI questions performed best when retrieval returned 3–5 well-structured chunks rather than many smaller ones.

* **Retrieval precision matters more than creativity:** Most observed failures were retrieval mismatches, not language-generation issues.

* **Explicitly stating knowledge limits improves safety:** Experiments that clearly said “the context does not support this” (e.g., appendicitis cured by medicine) were more trustworthy.

* **Certain topics are more retrieval-sensitive:** Differential-diagnosis and dermatology-style questions exposed weaknesses more than trauma or fracture care.

* **Moderate configurations yield the most stable answers:** Extreme settings (very small chunks, high k, or higher temperature) consistently degraded reliability.

**Recommended Best Choice**

* **Experiment 5**
* **Configuration:** chunk_size=800, overlap=100, k=5, temperature=0.1
* **Best overall stability** across all question types (protocols, symptoms, treatments, first aid).

* **Balanced retrieval:** Enough context to be complete, but not so much that irrelevant chunks leak in.

* **Strong grounding discipline:** Low temperature keeps answers factual while still readable.

* **Safer clinical behavior:** More likely to acknowledge when the source text does not support a claim.

* **Production-ready default:** Works well without requiring per-question tuning and minimizes high-risk hallucinations.


### Fine-tuning summary (5+ combinations)

| Experiment | Chunk size / overlap | Retriever | k | Temp | What improved | What got worse |
|---|---:|---|---:|---:|---|---|
| Exp 1 (Small chunks) | 512 / 64 | similarity | 4 | 0.0 | More precise snippets; good for direct facts | Can miss full protocol context; fragmented answers |
| Exp 2 (Large chunks) | 1500 / 200 | similarity | 3 | 0.0 | Better end-to-end clinical flow; fewer missing steps | More noise; may include unrelated details |
| Exp 3 (High retrieval) | 1024 / 128 | similarity | 6 | 0.0 | Higher recall; fewer omissions | Redundancy; can dilute relevance |
| Exp 4 (Higher creativity) | 1024 / 128 | similarity | 4 | 0.2 | Smoother narrative and transitions | Higher hallucination risk; weaker grounding |
| Exp 5 (Balanced) | 800 / 100 | similarity | 5 | 0.1 | Often best tradeoff of completeness + focus | Can still be mildly generic without strong prompting |

**Recommended starting point:** chunk size **800–1200**, overlap **100–150**, `k=5–6`, temperature **0.0–0.1**.


## Output Evaluation

### Define Judge Prompts and Templates

In [ ]:
# ---------------------------
# Output Evaluation
# ---------------------------

# Define the Judge Prompts
relevance_rater_system_message = """
You will be presented with a ###Question, the ###Context used by the AI system to generate a response, and the AI-generated ###Answer.

Your task is to judge the extent to which the ###Answer is relevant to the ###Question, considering whether it directly addresses the key aspects of the ###Question based on the provided ###Context.

Rate the relevance as follows:
- Rate 1 – The ###Answer is not relevant to the ###Question at all.
- Rate 2 – The ###Answer is only slightly relevant to the **###Question**, missing key aspects.
- Rate 3 – The ###Answer is moderately relevant, addressing some parts of the **###Question** but leaving out important details.
- Rate 4 – The ###Answer is mostly relevant, covering key aspects but with minor gaps.
- Rate 5 – The ###Answer is fully relevant, directly answering all important aspects of the **###Question** with appropriate details from the **###Context**.

The final output should be a single overall rating in the range of 1 to 5, along with a brief explanation of the rationale for the rating.
"""

groundedness_rater_system_message = """
You will be presented a ###Question, ###Context used by the AI system and AI generated ###Answer.

Your task is to judge the extent to which the ###Answer is derived from ###Context.

Rate it 1 - if The ###Answer is not derived from the ###Context at all
Rate it 2 - if The ###Answer is derived from the ###Context only to a limited extent
Rate it 3 - if The ###Answer is derived from ###Context to a good extent
Rate it 4 - if The ###Answer is derived from ###Context mostly
Rate it 5 - if The ###Answer is is derived from ###Context completely

The final output should be a single overall rating in the range of 1 to 5, along with a brief explanation of the rationale for the rating.
"""

# Define Input Template for the Judge
# This formats the data so the Judge knows what is what (###Question, etc.)
evaluation_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

### Define evaluation

In [ ]:
# ---------------------------
# 1. Define Helper Functions
# ---------------------------

def _safe_parse_json(text: str):
    """
    Robust JSON parsing. Attempts to find a JSON object within the text
    using regex if direct parsing fails.
    """
    text = text.strip()
    # Try direct JSON parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # If that fails, look for a JSON object pattern { ... }
        # The regex looks for the first opening brace and the last closing brace
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass

    # Fallback if parsing completely fails
    return {"rating": None, "rationale": "Failed to parse JSON: " + text[:100]}

judge_user_template = """
###Question:
{question}

###Context:
{context}

###Answer:
{answer}

Return STRICT JSON only (no extra text) with this format:
{{
  "rating": <integer 1-5>,
  "rationale": "<brief explanation>"
}}
"""

def llm_judge(system_prompt: str, question: str, context: str, answer: str,
              max_tokens=256, temperature=0.0):
    """
    Calls the LLM to judge the answer and returns a dict with rating+rationale.
    """
    user_prompt = judge_user_template.format(
        question=question,
        context=context,
        answer=answer
    )
    full_prompt = system_prompt.strip() + "\n\n" + user_prompt.strip()

    # Call LLM instance
    raw = llm(
        prompt=full_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.95,
        top_k=40,
        echo=False
    )
    text = raw["choices"][0]["text"].strip()
    return _safe_parse_json(text)

def evaluate_single_query(key, question, answer, context):
    """
    Evaluates a single query for Relevance and Groundedness.
    Returns a dictionary of results.
    """
    print(f"Evaluating {key}: {question[:60]}...")

    # Evaluate Relevance
    rel_result = llm_judge(
        system_prompt=relevance_rater_system_message,
        question=question,
        context=context,
        answer=answer
    )

    # Evaluate Groundedness
    grd_result = llm_judge(
        system_prompt=groundedness_rater_system_message,
        question=question,
        context=context,
        answer=answer
    )

    # Print immediate results
    print(f"  -> Relevance: {rel_result.get('rating')} | Rationale: {rel_result.get('rationale')[:50]}...")
    print(f"  -> Groundedness: {grd_result.get('rating')} | Rationale: {grd_result.get('rationale')[:50]}...")
    print("-" * 60)

    return {
        "question_key": key,
        "question": question,
        "relevance_rating": rel_result.get("rating"),
        "relevance_rationale": rel_result.get("rationale"),
        "groundedness_rating": grd_result.get("rating"),
        "groundedness_rationale": grd_result.get("rationale")
    }

### Run Evaluation

In [ ]:
# ---------------------------
# 2. Run Evaluation Loop
# ---------------------------

# Define your existing RAG data
rag_pack = {
    "Q1": {"question": query_1, "answer": rag_ans_1, "context": rag_ctx_1},
    "Q2": {"question": query_2, "answer": rag_ans_2, "context": rag_ctx_2},
    "Q3": {"question": query_3, "answer": rag_ans_3, "context": rag_ctx_3},
    "Q4": {"question": query_4, "answer": rag_ans_4, "context": rag_ctx_4},
    "Q5": {"question": query_5, "answer": rag_ans_5, "context": rag_ctx_5},
}

evaluation_results = []

print("Starting Individual Query Evaluation...\n" + "="*60)

for key, data in rag_pack.items():
    result = evaluate_single_query(
        key,
        data["question"],
        data["answer"],
        data["context"]
    )
    evaluation_results.append(result)

Starting Individual Query Evaluation...
Evaluating Q1: What is the protocol for managing sepsis in a critical care ...
  -> Relevance: 5 | Rationale: The answer is fully relevant to the question, prov...
  -> Groundedness: 4 | Rationale: The answer is derived from the context to a good e...
------------------------------------------------------------
Evaluating Q2: What are the common symptoms of appendicitis, and can it be ...
  -> Relevance: 5 | Rationale: The answer is fully relevant, directly answering a...
  -> Groundedness: 5 | Rationale: The answer is derived from the context completely....
------------------------------------------------------------
Evaluating Q3: What are the effective treatments or solutions for addressin...
  -> Relevance: 4 | Rationale: The answer is mostly relevant, covering key aspect...
  -> Groundedness: 3 | Rationale: The answer is derived from the context to a good e...
------------------------------------------------------------
Evaluating Q4: What t

### Evaluation Summary

In [ ]:
# ---------------------------
# 3. Final Summary & JSON Output
# ---------------------------

# Create DataFrame for display (similar to your screenshot)
eval_df = pd.DataFrame(evaluation_results)

print("\n### Final Evaluation Summary Table")
display(eval_df[["question_key", "relevance_rating", "groundedness_rating", "relevance_rationale", "groundedness_rationale"]])

# Calculate Averages
avg_rel = eval_df["relevance_rating"].mean()
avg_grd = eval_df["groundedness_rating"].mean()
print(f"\nAverage Relevance: {avg_rel:.2f}")
print(f"Average Groundedness: {avg_grd:.2f}")

# Convert to JSON for final storage
final_json_summary = json.dumps(evaluation_results, indent=2)
print("\n### Final JSON Output")
print(final_json_summary)


### Final Evaluation Summary Table


,question_key,relevance_rating,groundedness_rating,relevance_rationale,groundedness_rationale
0,Q1,5,4,"The answer is fully relevant to the question, ...",The answer is derived from the context to a go...
1,Q2,5,5,"The answer is fully relevant, directly answeri...",The answer is derived from the context complet...
2,Q3,4,3,"The answer is mostly relevant, covering key as...",The answer is derived from the context to a go...
3,Q4,5,4,"The answer is fully relevant to the question, ...",The answer is derived from the context to a go...
4,Q5,1,1,The provided context does not contain informat...,The provided context does not contain informat...



Average Relevance: 4.00
Average Groundedness: 3.40

### Final JSON Output
[
  {
    "question_key": "Q1",
    "question": "What is the protocol for managing sepsis in a critical care unit?",
    "relevance_rating": 5,
    "relevance_rationale": "The answer is fully relevant to the question, providing a comprehensive overview of the protocol for managing sepsis in a critical care unit, including key steps, monitoring, and targets.",
    "groundedness_rating": 4,
    "groundedness_rationale": "The answer is derived from the context to a good extent. The context provides a detailed overview of critical care medicine and the approach to managing critically ill patients. The answer also provides key steps and monitoring targets for managing sepsis in a critical care unit, which are relevant to the context. However, the answer could be improved by providing more specific details and examples to illustrate each step and target."
  },
  {
    "question_key": "Q2",
    "question": "What are th

## Actionable Insights and Business Recommendations

### Key takeaways (from LLM, Prompt Engineering, and RAG experiments)
- **Baseline LLM** responses are generally coherent, but can be **generic** and may include details that are not verifiable from the provided medical reference.
- **Prompt engineering** improves *readability and structure* (e.g., checklists, red flags), but by itself it **does not guarantee grounding** in an authoritative source.
- **RAG** improves **groundedness** and reduces hallucination risk by anchoring responses to the Merck Manual context. The best outputs are achieved when the prompt *forces structure* and the retriever returns **relevant, non-redundant** chunks.
- **Chunking + retrieval** is a major quality lever:
  - Smaller chunks increase precision but may fragment clinical protocols.
  - Larger chunks increase completeness but can add noise.
  - Increasing `k` improves recall but can reduce relevance if the context becomes too broad.
- **LLM-as-a-judge** is useful for regression testing across configurations, but results must be **repeatable** (low temperature, structured JSON outputs, consistent context).

### Business recommendations
- **Deploy as clinical decision-support (not autonomous diagnosis):**
  - Position the system as a *reference and summarization assistant* that helps clinicians quickly extract protocol-like steps from approved sources.
- **Traceability by design:**
  - Display the answer alongside the top retrieved excerpts (and page/section metadata when available) to support clinician trust and auditability.
- **Governance and safety:**
  - Add UI disclaimers (“informational support only; clinician remains responsible”).
  - Log question, retrieved chunk IDs, answer, and evaluation scores for audits.
  - Establish a refresh cadence for the knowledge base (clinical guidance evolves).
- **Quality monitoring:**
  - Use relevance + groundedness scores as automated regression checks whenever you change the embedding model, chunking strategy, retriever type, or LLM.
  - Add periodic human review for high-risk topics (ICU, surgery, pediatrics).

### Next steps
- Expand the corpus with **institution-specific protocols** (if allowed) and compare scores against the Merck-only baseline.
- Add **MMR retrieval** and citation formatting to reduce redundancy and improve explainability.
- Create a small “gold set” of questions for human evaluation to validate LLM-judge scores.
